## Problem 05: LangGraph를 활용한 리뷰 응대 에이전트 만들기
문제 설명
고객의 상품 리뷰를 분석하여 '긍정'인지 '부정'인지 판단하고, 그 결과에 따라 알맞은 답변을 생성하는 조건부 라우팅(Conditional Routing) 워크플로우를 만들려고 합니다.

API 키 연결 에러를 방지하기 위해 이번 실습에서는 복잡한 LLM 대신 미리 준비된 간단한 파이썬 함수를 노드(Node)로 사용합니다.

요구사항:

StateGraph에 들어갈 상태(State) 클래스가 이미 준비되어 있습니다. 이를 바탕으로 그래프 workflow를 생성하세요.

세 가지 함수(analyze_sentiment, positive_reply, negative_reply)를 각각 그래프의 **노드(Node)**로 추가하세요. (이름은 자유롭게 지어도 됩니다)

START에서 분석 노드로 가는 시작 **엣지(Edge)**를 연결하세요.

분석 노드 다음에는 route_sentiment 함수를 사용하는 **조건부 엣지(Conditional Edges)**를 연결하여, 긍정/부정에 따라 다른 노드로 가도록 길을 나누세요.

긍정 노드와 부정 노드가 끝나면 각각 END로 가도록 엣지를 연결하고, 그래프를 컴파일(compile())하여 실행해 보세요!

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 그래프가 기억할 데이터 공간 (상태, State) 정의
class ReviewState(TypedDict):
    review: str      # 고객의 원래 리뷰
    sentiment: str   # 긍정/부정 분석 결과
    reply: str       # 최종 답변

# --- 노드(Node)와 라우팅에 사용될 함수들 (이 부분은 수정하지 마세요) ---
def analyze_sentiment(state: ReviewState):
    """리뷰 텍스트를 분석하여 긍정/부정을 판단하는 함수"""
    review = state["review"]
    if "좋" in review or "최고" in review or "완벽" in review:
        return {"sentiment": "positive"}
    else:
        return {"sentiment": "negative"}

def positive_reply(state: ReviewState):
    """긍정적인 리뷰에 대한 답변 생성 함수"""
    return {"reply": "감사합니다! 앞으로도 더 좋은 서비스로 보답하겠습니다."}

def negative_reply(state: ReviewState):
    """부정적인 리뷰에 대한 답변 생성 함수"""
    return {"reply": "불편을 드려 죄송합니다. 즉시 개선될 수 있도록 조치하겠습니다."}

def route_sentiment(state: ReviewState):
    """sentiment 값에 따라 다음으로 이동할 노드 이름을 결정하는 함수"""
    if state["sentiment"] == "positive":
        return "positive_node" # 아래 노드 추가 시 지정할 이름과 맞춰주세요
    else:
        return "negative_node" # 아래 노드 추가 시 지정할 이름과 맞춰주세요
# -------------------------------------------------------------------------

# 2. StateGraph 생성 (ReviewState를 기반으로 생성)
# 여기에 코드를 작성하세요.
graph = StateGraph(ReviewState)

# 3. 노드(Node) 추가하기 (이름은 "analyze_node", "positive_node", "negative_node" 로 지정하세요)
# 여기에 코드를 작성하세요.
graph.add_node("analyze_node", analyze_sentiment)
graph.add_node("positive_node", positive_reply)
graph.add_node("negative_node", negative_reply)

# 4. 엣지(Edge) 연결하기 (START -> analyze_node -> 조건부 분기 -> END)
graph.add_edge(START, "analyze_node")
graph.add_conditional_edges("analyze_node", route_sentiment)
graph.add_edge("positive_node", END)
graph.add_edge("negative_node", END)

# 여기에 코드를 작성하세요.
# 힌트 1: 시작 엣지 연결: workflow.add_edge(START, "첫노드이름")
# 힌트 2: 조건부 엣지 연결: workflow.add_conditional_edges("분기할노드이름", route_sentiment)
# 힌트 3: 종료 엣지 연결: workflow.add_edge("노드이름", END)


# 5. 그래프 컴파일 (조립 완료)
# 여기에 코드를 작성하세요. (결과를 app 변수에 저장하세요)
app = graph.compile()

# --- 테스트 코드 (수정하지 말고 그대로 실행해 보세요) ---
input_data_1 = {"review": "어제 산 마우스 너무 가볍고 클릭감도 좋네요. 완벽합니다!"}
input_data_2 = {"review": "포장 상태가 너무 엉망입니다. 실망했어요."}

print("--- 1번 고객 리뷰 결과 ---")
print(app.invoke(input_data_1))

print("\n--- 2번 고객 리뷰 결과 ---")
print(app.invoke(input_data_2))

--- 1번 고객 리뷰 결과 ---
{'review': '어제 산 마우스 너무 가볍고 클릭감도 좋네요. 완벽합니다!', 'sentiment': 'positive', 'reply': '감사합니다! 앞으로도 더 좋은 서비스로 보답하겠습니다.'}

--- 2번 고객 리뷰 결과 ---
{'review': '포장 상태가 너무 엉망입니다. 실망했어요.', 'sentiment': 'negative', 'reply': '불편을 드려 죄송합니다. 즉시 개선될 수 있도록 조치하겠습니다.'}
